# LAlatex aligned guide

This notebook documents `aligned(...)`, the structured helper for derivations, equation chains, and aligned linear algebra statements.

Use `aligned` when you want LAlatex to build a LaTeX `aligned` environment from row cells. You provide the cells in each row; LAlatex renders each cell with the normal `L_show` policy and inserts LaTeX alignment markers between cells.

In [ ]:
using LAlatex
using LaTeXStrings

set_backend!(:symbolics)
@syms x y

## Row forms

`aligned` accepts three row forms:

- vector rows, such as `[lhs, relation, rhs]`
- tuple rows, such as `(lhs, relation, rhs)`
- pair rows, such as `lhs => rhs`

Vector and tuple rows can have any positive number of cells. Pair rows are shorthand for equality rows.

In [ ]:
display(l_show(
    aligned(
        [L"Ax", L"=", [x, y]],
        (L"x", L"\in", L"\mathcal{N}(A)"),
        L"\dim\mathcal{N}(A)" => L"n - \operatorname{rank}(A)",
    );
    arraystyle=:bmatrix,
))

## Implicit alignment markers

For vector and tuple rows, LAlatex joins cells like this:

```text
cell_1 & cell_2 & cell_3 \\
```

Do not put `&` in the row cells. Pass each visual column as a separate Julia value.

## Pair rows

`aligned(left => right)` is equivalent to `aligned((left, L"=", right))`. Use pair rows for simple equality chains. Use vector or tuple rows when you need another relation such as `\in`, `\le`, `\Rightarrow`, or multiple aligned columns.

In [ ]:
display(l_show(
    aligned(
        L"u + v" => [x + 1, y - 1],
        [L"u - v", L"=", [x - 1, y + 1]],
    );
    arraystyle=:pmatrix,
))

## Text versus math cells

Plain Julia strings are rendered as text. Use `L"..."` or `LaTeXString` for relation symbols and raw LaTeX math. This matters most for cells such as `=`, `\in`, `\Rightarrow`, and named spaces.

In [ ]:
display(l_show(
    aligned(
        ["text label", L"=", x + y],
        [L"x", L"\in", L"\mathbb{R}"],
    ),
))

## Propagated display options

`aligned` does not have separate formatting keywords of its own. It propagates the relevant `L_show` options into every row cell:

- `arraystyle`: vector and matrix delimiters inside row cells
- `number_formatter`: scalar number formatting inside row cells
- `per_element_style`: matrix entry styling inside row cells
- `factor_out`: denominator factoring for array cells
- `symopts`: display-time symbolic transforms
- `color`: color wrapper for each rendered cell

In [ ]:
display(l_show(
    aligned(
        [(x + y)^2, L"=", x^2 + 2x*y + y^2],
        [[1//2, 1//3], L"=", L"r"],
    );
    symopts=(expand=true,),
    arraystyle=:bmatrix,
    number_formatter=n -> n isa Integer ? LaTeXString("\\mathbf{$n}") : n,
))

## Invalid rows

Rows must be pairs, tuples, or vectors. Empty vector and tuple rows are rejected because an aligned row must have at least one cell.

In [ ]:
# These throw ArgumentError:
# L_show(aligned(x))
# L_show(aligned([]))

## Raw LaTeX fallback

`aligned` is for structured rows where LAlatex owns the `&` markers. If you need hand-written alignment markers or unusual row separators, pass a complete `LaTeXString` directly to `L_show` instead of using `aligned`.

## Summary

Use vector or tuple rows when you want explicit columns. Use pair rows for equality. Let `aligned` insert `&` markers. Use `LaTeXString`s for relation symbols and raw math, and use normal `L_show` keywords to control row-cell rendering.